# Load data

In [2]:
import pandas as pd
df = df = pd.read_csv("transactions.csv")
print(df.head())

                narration        mode    type    category  subcategory  \
0    FD interest credited        NEFT  Credit  investment  fd_interest   
1               FD INT CR        NEFT  Credit  investment  fd_interest   
2  Fixed deposit interest        NEFT  Credit  investment  fd_interest   
3              FD booking  NETBANKING   Debit  investment   fd_booking   
4          Open FD online  NETBANKING   Debit  investment   fd_booking   

   Unnamed: 5  Unnamed: 6  Unnamed: 7  
0         NaN         NaN         NaN  
1         NaN         NaN         NaN  
2         NaN         NaN         NaN  
3         NaN         NaN         NaN  
4         NaN         NaN         NaN  


# Create input_text

In [ ]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [ ]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [ ]:
df["label"] = df["category"] + "/" + df["subcategory"]

In [ ]:
print(df[["input_text", "label"]].head())

# Check label distribution

In [ ]:
print(df["label"].value_counts())

# Define X and y

In [ ]:
X = df["input_text"]   
y = df["label"]       

# Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # reproducibility
)

# Model Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Training

In [ ]:
model.fit(X_train, y_train)

## Prediction

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import f1_score, fbeta_score, classification_report

# F1 Score (balanced)
f1 = f1_score(y_test, y_pred, average='weighted')

# F2 Score (recall-focused)
f2 = fbeta_score(y_test, y_pred, beta=2, average='weighted')

print("F1 Score:", round(f1, 3))
print("F2 Score:", round(f2, 3))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

## Accuracy

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
print(df[["narration", "label"]])

In [ ]:
print(df["label"].value_counts())

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

In [ ]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

for i, label in enumerate(classifier.classes_):
    top_features = sorted(
        zip(classifier.coef_[i], feature_names),
        reverse=True
    )[:5]
    
    print(f"\nTop words for {label}:")
    for coef, word in top_features:
        print(word)

## Debug

In [ ]:
for i in range(len(X_test)):
    print("Input:", X_test.iloc[i])
    print("Actual:", y_test.iloc[i])
    print("Predicted:", y_pred[i])
    print("-----")

## Confidence

In [ ]:
probs = model.predict_proba(X_test)
for i in range(len(X_test)):
    print("Prediction:", model.classes_[probs[i].argmax()])
    print("Confidence:", round(probs[i].max(), 2))

In [ ]:
def confidence_level(conf):
    if conf > 0.75:
        return "High"
    elif conf > 0.5:
        return "Medium"
    else:
        return "Low"

In [ ]:
from sklearn.metrics import classification_report, f1_score

print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

# Output

In [ ]:
results = []
for i in range(len(df)):
    input_text = df["input_text"].iloc[i]
    probs = model.predict_proba([input_text])[0]
    pred_label = model.classes_[probs.argmax()]
    confidence = probs.max()
    category, subcategory = pred_label.split("/", 1)
    results.append({
    "narration": df["narration"].iloc[i],
    "mode": df["mode"].iloc[i],
    "type": df["type"].iloc[i],
    "ground_truth": f"{category}/{subcategory}",
    "confidence": round(confidence, 2),
    "confidence_percent": f"{round(confidence*100, 1)}%",
    "confidence_level": confidence_level(confidence)
})
result_df = pd.DataFrame(results)
print(result_df.to_string())